# 🟠 Lesson 20 — PyGMT

**Level: Advanced** · The Generic Mapping Tools in Python — the gold standard for seismology/geodesy figures.

*Part of the companion package for [python_for_geologists](https://github.com/kevinalexandr19/python_for_geologists) by Kevin Alexander Gomez.*

> ⚠️ **This notebook is not executed in this repo** — PyGMT needs the GMT C library, which is best installed with conda: The code is complete and ready to run once you have set it up.
```bash
conda install -c conda-forge pygmt
```

In [ ]:
import pygmt
print(pygmt.show_versions())

## 1. A shaded-relief map from built-in Earth data
PyGMT downloads tiles of global relief automatically.

In [ ]:
region = [-84, -66, -20, 2]          # Peru

fig = pygmt.Figure()
grid = pygmt.datasets.load_earth_relief(resolution="03m", region=region)
fig.grdimage(grid=grid, projection="M15c", cmap="geo", shading=True)
fig.coast(shorelines="1/0.5p,black", borders="1/0.5p,gray40")
fig.colorbar(frame='af+l"elevation (m)"')
fig.basemap(frame=True)
fig.show()

## 2. Plot the earthquake catalogue on top

In [ ]:
import pandas as pd
from pathlib import Path

q = pd.read_csv(Path("..") / "data" / "earthquakes.csv")
q.columns = ["date", "time", "lat", "lon", "depth_km", "magnitude"]

fig = pygmt.Figure()
fig.basemap(region=region, projection="M15c", frame=True)
fig.coast(land="gray95", shorelines="1/0.5p,black")
pygmt.makecpt(cmap="viridis", series=[0, 700], reverse=True)
fig.plot(x=q["lon"], y=q["lat"],
         size=0.02 * (2 ** q["magnitude"] / 10),
         fill=q["depth_km"], cmap=True, style="cc", pen="0.2p,black")
fig.colorbar(frame='af+l"depth (km)"')
fig.show()

## 3. Topographic profile across the subduction zone

In [ ]:
points = pygmt.project(center=[-78, -10], endpoint=[-70, -14], generate=10)
track = pygmt.grdtrack(points=points, grid=grid, newcolname="elev")

fig = pygmt.Figure()
fig.plot(x=track["p"], y=track["elev"], pen="1.5p,firebrick",
         region=[0, track["p"].max(), -7000, 6000],
         projection="X15c/6c",
         frame=['xaf+l"distance (km)"', 'yaf+l"elevation (m)"', "WSen"])
fig.show()

## 4. Save publication output

In [ ]:
fig.savefig("outputs/peru_profile.pdf")   # also .png, .eps ...

### ✏️ Try it
1. Zoom `region` to Arequipa and use `resolution='30s'` relief for detail.
2. Colour the profile by slope, or add the deep earthquakes projected onto the section.

📚 Docs: https://www.pygmt.org/latest/